# Logistic Regression

Despite its name, logistic regression is a **classification** algorithm. It models the probability of class membership using the logistic (sigmoid) function.

1. **Binary Classification** - Sigmoid function, decision boundary, probability calibration
2. **Multi-class Classification** - One-vs-Rest and Softmax
3. **Regularization** - L1, L2, and their effects
4. **Threshold Tuning** - Optimizing for business objectives

**Dataset**: Breast Cancer Wisconsin (binary), Iris (multi-class)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, roc_curve, roc_auc_score,
    precision_recall_curve, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")

## 1. Binary Classification

The sigmoid function maps any real number to (0, 1):

$$P(y=1|\mathbf{x}) = \sigma(\mathbf{w}^T\mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T\mathbf{x} + b)}}$$

In [ ]:
# Visualize the sigmoid function
z = np.linspace(-6, 6, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid, color="teal", lw=2)
plt.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
plt.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("z = w^T x + b")
plt.ylabel("P(y=1)")
plt.title("Sigmoid Function")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Load and train
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=1.0, max_iter=5000, random_state=42)),
])
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=data.target_names))

## 2. Threshold Tuning

The default threshold of 0.5 isn't always optimal. In medical diagnosis, we might prefer higher recall (catching all positives) even at the cost of precision.

In [ ]:
# Analyze threshold impact
thresholds = np.arange(0.1, 0.9, 0.05)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score, f1_score
    precisions.append(precision_score(y_test, y_t, zero_division=0))
    recalls.append(recall_score(y_test, y_t))
    f1s.append(f1_score(y_test, y_t))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions, label="Precision", color="teal")
plt.plot(thresholds, recalls, label="Recall", color="coral")
plt.plot(thresholds, f1s, label="F1 Score", color="steelblue", linewidth=2)
plt.axvline(x=0.5, color="gray", linestyle="--", label="Default threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall, F1 vs. Threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Multi-class Classification

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris)

# Multinomial logistic regression (softmax)
pipe_multi = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=5000)),
])
pipe_multi.fit(X_tr, y_tr)

print(classification_report(y_te, pipe_multi.predict(X_te), target_names=iris.target_names))

# Show probability calibration - softmax outputs sum to 1
sample_probs = pipe_multi.predict_proba(X_te[:5])
print("\nSample predicted probabilities (sum to 1):")
print(pd.DataFrame(sample_probs, columns=iris.target_names).round(3))

## 4. Effect of Regularization Strength

In [ ]:
# C = 1/lambda: smaller C = stronger regularization
C_values = np.logspace(-3, 3, 20)
train_accs, test_accs = [], []

for C in C_values:
    pipe_c = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C, max_iter=5000, random_state=42)),
    ])
    pipe_c.fit(X_train, y_train)
    train_accs.append(pipe_c.score(X_train, y_train))
    test_accs.append(pipe_c.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.semilogx(C_values, train_accs, "o-", label="Train", color="teal")
plt.semilogx(C_values, test_accs, "o-", label="Test", color="coral")
plt.xlabel("C (inverse regularization strength)")
plt.ylabel("Accuracy")
plt.title("Effect of Regularization on Logistic Regression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Logistic regression outputs calibrated probabilities** - not just class labels
2. **Tune the decision threshold** based on your business objective (precision vs recall)
3. **C controls regularization** - lower C = stronger regularization = simpler model
4. **For multi-class**: use `multinomial` (softmax) rather than one-vs-rest when classes are mutually exclusive
5. **Fast, interpretable, and often competitive** - always try logistic regression as a baseline